In [62]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

In [63]:

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive"
    }

job_titles = []
company_names= []
locations = []
salary = []
experience_list = []
skills_required= []
job_description = []
posted_dates = []
job_links = []

for j in range(1,16):
    url = f"https://internshala.com/jobs/analytics,data-science,machine-learning-jobs/page-{j}/"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text , "html.parser")
      
    #extracting job titles
    for title in soup.find_all("a" , class_ = "job-title-href"):
        job_titles.append(title.text.strip())
    
    #extracting company names
    for company in soup.find_all("p" , class_ = "company-name"):
        company_names.append(company.text.strip())  
    
    #extracting locations
    for loc in soup.find_all("div", class_="individual_internship"):
        location_tag = loc.select_one("p.row-1-item.locations span a")
        if location_tag:
            locations.append(location_tag.get_text(strip=True)) 
        else:
            locations.append(None)
    
    #extracting salary
    for job in soup.find_all("div", class_="individual_internship"):
        salary_tag = job.select_one("i.ic-16-money + span")
    
        if salary_tag:
            salary.append(salary_tag.get_text(strip=True))
        else:
            salary.append(None)
       
    #extracting experience
    for exp in soup.find_all("div", class_="individual_internship_details"):
        exp_tag = exp.select_one("i.ic-16-briefcase + span")
        if exp_tag:
            experience = exp_tag.get_text(strip=True)
            experience_list.append(experience)
        else:
            experience_list.append(None)
    
    #extracting skills required
    for job in soup.find_all("div", class_="individual_internship"):
        skill_tags = job.select("div.job_skills div.job_skill")
        skills = [skill.get_text(strip=True) for skill in skill_tags]
        skills_required.append(skills)
    
    #extracting job description
    for des in soup.find_all("div", class_="individual_internship_details"):
        des_tag = des.select_one("div.about_job div.text")
        if des_tag:
            description = des_tag.get_text(strip=True)
        job_description.append(description)
    
    #extracting posted dates
    for job in soup.find_all("div", class_="internship_meta"):
        date = job.select_one("div.detail-row-2 div.color-labels span")
        posted_dates.append(date.text.strip() if date else None)
    
    #extracting job links
    for link in soup.find_all("div", class_="internship_meta"):
        link_tag = link.select_one('a.job-title-href')
        if link_tag:
            job_links.append("https://internshala.com" + link_tag['href'])

time.sleep(1)

In [64]:
print(len(job_titles))
print(len(company_names))
print(len(locations))
print(len(salary))
print(len(experience_list))
print(len(skills_required))
print(len(job_description))
print(len(posted_dates))
print(len(job_links))

610
610
610
610
610
610
610
610
610


Creating DataFrame

In [65]:
data = pd.DataFrame({
    "Job Title": job_titles,
    "Company Name": company_names,
    "Location": locations,
    "Experience Required": experience_list,
    "Salary" : salary,
    "Skills Required": skills_required,
    "Job Description": job_description,
    "Posted Date": posted_dates,
    "Scrape Date": pd.Timestamp.now().date(),
    "Job Link" : job_links ,
    "Source Platform": "Internshala" , 
    
    })

In [66]:
data.head()

,Job Title,Company Name,Location,Experience Required,Salary,Skills Required,Job Description,Posted Date,Scrape Date,Job Link,Source Platform
0,Telegram Trader,Trading Fox Private Limited,Jaipur,No experience required,"₹ 2,00,000 - 2,50,000","[Data Analytics, Technical Analysis, Equity Re...",As a Telegram Trader at Trading Fox Private Li...,2 weeks ago,2026-04-28,https://internshala.com/job/detail/fresher-tel...,Internshala
1,Junior Search Engine Optimization (SEO) Executive,IWrite India,Delhi,1 year(s),"₹ 3,00,000 - 4,00,000","[Google Analytics, Search Engine Marketing (SE...","As a Junior SEO Executive at IWrite India, you...",2 days ago,2026-04-28,https://internshala.com/job/detail/junior-sear...,Internshala
2,Robotics Educator,Skill Bharat Association,Mumbai,1 year(s),"₹ 2,00,000 - 4,50,000","[Python, Robotics, Teaching, Community Managem...",Key Responsibilities:\r\n\r\n1. Conduct roboti...,2 weeks ago,2026-04-28,https://internshala.com/job/detail/robotics-ed...,Internshala
3,Artificial Intelligence (AI) Executive,Tidal7 Asia - A J7 Agency,Mumbai,1 year(s),"₹ 2,00,000 - 4,00,000","[Generative AI Development, Generative AI Tool...",Tidal7 (a J7 agency) is at the forefront of th...,1 week ago,2026-04-28,https://internshala.com/job/detail/artificial-...,Internshala
4,Robotics Engineer,Playto Labs,Bangalore,No experience required,"₹ 3,00,000 - 3,25,000","[Python, ARM Microcontroller, Embedded Systems...",Key Responsibilities:\r\n\r\n1. Work on roboti...,1 day ago,2026-04-28,https://internshala.com/job/detail/fresher-rob...,Internshala


In [67]:
data.shape

(610, 11)

In [80]:
relevant_keywords = [
    "data analyst",
    "data scientist",
    "business analyst",
    "machine learning",
    "data engineer",
    "product analyst",
    "analytics",
    "bi",
    "etl"
]

# Create regex pattern
pattern = "|".join(re.escape(keyword) for keyword in relevant_keywords)

# Filter rows based on job title
df = data[data["Job Title"].str.contains(pattern, case=False, na=False)]

df.reset_index(drop=True, inplace=True)
df.head()

,Job Title,Company Name,Location,Experience Required,Salary,Skills Required,Job Description,Posted Date,Scrape Date,Job Link,Source Platform
0,Punjabi-Speaking Online Data Analyst,"TELUS International AI Inc. (Las Vegas, United...",Work from home,No experience required,"₹ 4,50,000 - 4,60,000","[English Proficiency (Spoken), English Profici...",Are you a detail-oriented individual with a pa...,6 days ago,2026-04-28,https://internshala.com/job/detail/fresher-rem...,Internshala
1,Malayalam Language Data Analyst For Online,"TELUS International AI Inc. (Las Vegas, United...",Work from home,No experience required,"₹ 4,50,000 - 4,60,000",[Data Analytics],A Day in the Life of an Online Data Analyst: I...,3 weeks ago,2026-04-28,https://internshala.com/job/detail/fresher-rem...,Internshala
2,Online Data Analyst – Marathi Speaker,"TELUS International AI Inc. (Las Vegas, United...",Work from home,No experience required,"₹ 4,50,000 - 4,60,000",[Data Analytics],Are you a detail-oriented individual with a pa...,3 weeks ago,2026-04-28,https://internshala.com/job/detail/fresher-rem...,Internshala
3,Online Data Analyst - Gujarati Language,"TELUS International AI Inc. (Las Vegas, United...",Work from home,No experience required,"₹ 4,50,000 - 4,60,000",[Data Analysis],A Day in the Life of an Online Data Analyst:\r...,3 weeks ago,2026-04-28,https://internshala.com/job/detail/fresher-rem...,Internshala
4,Remote Online Data Analyst (Urdu-Speaking),"TELUS International AI Inc. (Las Vegas, United...",Work from home,No experience required,"₹ 4,50,000 - 4,60,000","[Urdu Proficiency(Spoken), Urdu Proficiency(Wr...",Key Responsibilities: \r\n\r\n1. Work on a pro...,3 weeks ago,2026-04-28,https://internshala.com/job/detail/fresher-rem...,Internshala


In [81]:
df.shape

(166, 11)

In [79]:
df["Job Title"].values

array(['Artificial Intelligence (AI) Executive', 'AI Generalist',
       'AI Agent And Tool Associate', 'Robotics Trainer',
       'Artificial Intelligence (AI) Executive',
       'Artificial Intelligence (AI) Executive',
       'Management Trainee - Prompt Engineering',
       'Punjabi-Speaking Online Data Analyst',
       'Malayalam Language Data Analyst For Online',
       'Online Data Analyst – Marathi Speaker',
       'Online Data Analyst - Gujarati Language',
       'Remote Online Data Analyst (Urdu-Speaking)',
       'Remote Online Data Analyst (Tamil-Speaking)',
       'Big Data Associate', 'Data Annotator', 'Senior Data Engineer',
       'Data/Analytics Lead',
       'Senior Frontend Engineer & AI-Powered Interactive Design',
       'Senior ETL Consultant',
       'SAP Controller & AI Executive (For Mines Industry)',
       'Generative AI Architect (Code Generation)', 'Gen AI Engineer',
       'Junior Data Scientist', 'AI Developer',
       'Digital Market Intelligence Analyst

In [85]:
import os

print(os.path.exists(r"d:\Startup\Project\ai-career-coach\data\data\raw"))
print(os.path.exists(r"d:\Startup\Project\ai-career-coach\data\data\processed"))

True
True


In [83]:



print(os.listdir(r"d:\Startup\Project\ai-career-coach"))

['.git', '.gitignore', 'app', 'data', 'main.py', 'notebooks', 'README.md', 'requirements.txt', 'src', 'venv']


In [84]:
print(os.listdir(r"d:\Startup\Project\ai-career-coach\data"))

['data']
